# Setup · Run the study on a GCP H100 (from your laptop)

This notebook does everything we did by hand, in order, and explains each step:

| Step | What happens | Where it runs |
|---|---|---|
| 1. Configure | Pick the VM, the **vLLM engine arguments**, and which measurements to take | Laptop |
| 2. Preflight | Check `gcloud` login, project and GPU/CPU quotas | Laptop |
| 3. Launch VM | Create the Spot `a3-highgpu-1g` (1x H100) or start the existing one; arm an auto-shutdown timer | GCP |
| 4. Upload + setup | Copy `scripts/` + this run's config; install compiler, uv, venv, pinned vLLM, download the model (idempotent) | VM |
| 5. Environment capture | Driver, CUDA, clocks, every package version, model revision → `env/` | VM |
| 6. Benchmark sweep | `vllm bench latency` over batch sizes × output lengths | VM (H100) |
| 7. Nsight Systems | Timelines of `prof_run.py`: CUDA graphs vs eager, B=1 / B=32 | VM (H100) |
| 8. Nsight Compute | The ncu plan from notebook 02 (or a light metric sweep) | VM (H100) |
| 9. Transfer | tar → copy → **sha256-verify every file** on your laptop | VM → laptop |
| 10. Shut down | Stop (default) or delete the VM; print the GPU time used | GCP |
| 11. Analyze | Run notebook 02 on this run | Laptop |

**How the pieces fit:** this notebook's Python kernel runs on your laptop and drives the VM with the `gcloud` CLI (every command is printed). Long GPU jobs run on the VM inside `tmux` (`scripts/job.sh`), so a dropped connection or a sleeping laptop never kills a measurement; the notebook polls every minute. Every stage skips work that is already done, so after a Spot preemption the notebook restarts the VM and resumes automatically.

**Measurement integrity:** all timing is taken on the VM (vLLM's timers, nsys/ncu GPU timestamps and counters). Nothing measured here crosses the network. Don't keep a VS Code Remote-SSH window open on the VM while stages run; its server competes for the CPU that vLLM's scheduler uses.

> **Cost safety:** the VM bills while it is RUNNING. Step 3 arms a self-shutdown timer on the VM, and section 12 runs the whole pipeline with stop-in-`finally`. If you step through cells manually and stop halfway, run the **Shut down** cell (step 10).

## 0. Prerequisites (one time, on your laptop)
```bash
# install the gcloud CLI first: https://cloud.google.com/sdk/docs/install
gcloud auth login                           # browser login
bash setup_laptop.sh                        # creates .venv with requirements.txt and registers the Jupyter kernel
```
Then pick the **sillybench** kernel for this notebook in VS Code. Quotas needed in the project (a new project starts at 0; request in Console → IAM & Admin → Quotas):
`Preemptible NVIDIA H100 GPUs` (region) ≥ 1, `GPUs (all regions)` ≥ 1, `CPUs (all regions)` ≥ 26.

## 1. Configure
### 1a. Where to run

In [ ]:
import datetime, json, sys, time
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))

PROJECT   = "h100-qwen27b"                 # GCP project ID (lowercase ID, not the display name)
REGION    = "us-central1"                  # where your H100 quota is
ZONES     = ["us-central1-a", "us-central1-b", "us-central1-c"]   # tried in order when creating
VM_NAME   = "h100-prof"
MACHINE   = "a3-highgpu-1g"                # 1x H100 80GB HBM3, 26 vCPU, 234 GB RAM
IMAGE_FAMILY  = "common-cu129-ubuntu-2204-nvidia-580"   # Deep Learning VM image: driver + CUDA + nsys/ncu
IMAGE_PROJECT = "deeplearning-platform-release"
BOOT_DISK_GB  = 240                        # model (~52 GB) + venv + traces
SPOT = True                                # 1-GPU A3 machines are Spot / Flex-start only on GCP

SHUTDOWN_TIMER_MIN = 240                   # VM powers itself off after this many minutes, no matter what
FINAL_ACTION = "stop"                      # after verified transfer: "stop" (recommended: keeps disk + setup, ~$0.80/day for 240 GB) | "delete" | "keep"
                                           # H100 quota is a project limit: neither stop nor delete gives it back
ON_FAILURE   = "stop"                      # a stage fails: "stop" = pull what exists + stop the VM | "keep" = leave it running to debug
AUTO_RESUME  = True                        # restart + resume automatically after a Spot preemption
POLL_SECONDS = 60
PRICE_PER_HOUR_USD = None                  # optional: your Spot price for a3-highgpu-1g, for the cost estimate

### 1b. vLLM engine arguments (edit here)
These are the exact arguments for **every** measurement in this run: `vllm bench latency` receives them as CLI flags (`max_num_seqs` → `--max-num-seqs`), and `scripts/prof_run.py` passes them to `vllm.LLM(**VLLM)` for the nsys/ncu captures. Booleans become flags (`True` → `--flag`; `False` → omitted, or `--no-flag` for prefix caching). Any other vLLM engine argument can be added as a new key.

Why these defaults (all found the hard way on 2026-10-09):
* `language_model_only=True`: Qwen3.8-27B is a multimodal checkpoint; we profile text only.
* `max_num_seqs=256`: the default 1024 fails because only ~358 GDN recurrent-state blocks fit in memory.
* `enable_prefix_caching=False`: synthetic prompts repeat tokens; caching would skip prefill and corrupt the measurement.

In [ ]:
VLLM = {
    "model": "Qwen/Qwen3.8-27B",          # e.g. "Qwen/Qwen3.8-27B-FP8" for the FP8 study
    "language_model_only": True,
    "max_model_len": 8192,
    "max_num_seqs": 256,
    "gpu_memory_utilization": 0.92,
    "enable_prefix_caching": False,
    # "enforce_eager": False,            # nsys stage sets graphs/eager per capture; leave unset here
    # "kv_cache_dtype": "fp8",
    # "dtype": "bfloat16",
    # "max_num_batched_tokens": 16384,
    # "compilation_config": {"cudagraph_mode": "FULL_AND_PIECEWISE"},
}
VLLM_EXTRA_CLI = []                        # raw extra flags for `vllm bench latency` only, e.g. ["--gdn-prefill-backend", "triton"]
ENV_VARS = {}                              # environment variables for every job, e.g. {"VLLM_USE_FLASHINFER_SAMPLER": "0"}

### 1c. What to measure
* **bench**: decode time per token = (latency(out=128) − latency(out=1)) / 127 at each batch size.
* **nsys**: one capture per (batch, mode). `out_tokens` = forward passes in the measured range. `gpu_metrics=True` adds one B=1 capture with sampled HBM bandwidth / SM activity (needs sudo; optional).
* **ncu**: if notebook 02 has written `runs/<RUN_ID>/ncu_plan/ncu_commands.sh`, that plan runs (deep dives on the kernels it picked); otherwise a light metric-only sweep of a short decode range.

Typical sessions: **Session 1** = `["setup", "env", "bench", "nsys"]` → run notebook 02 → **Session 2** with the same `RUN_ID` = `["setup", "env", "ncu"]` (setup is a no-op when already done).

In [ ]:
STAGES = ["setup", "env", "bench", "nsys", "ncu"]      # remove what you don't want this session

MODEL_TAG = VLLM["model"].split("/")[-1].lower()
RUN_ID = None                                          # None = new run; or reuse e.g. "2026-10-14_qwen3.8-27b_bf16"
RUN_TAG = "bf16"
RUN_ID = RUN_ID or f"{datetime.date.today():%Y-%m-%d}_{MODEL_TAG}_{RUN_TAG}"

WORKLOAD = {"input_len": 512}
BENCH = {"enabled": "bench" in STAGES, "batch_sizes": [1, 8, 32, 64, 128, 256],
         "output_lens": [1, 128], "warmup_iters": 3, "iters": 5}
NSYS = {"enabled": "nsys" in STAGES, "batch_sizes": [1, 32], "modes": ["graphs", "eager"], "out_tokens": 64,
        "trace": "cuda,nvtx,osrt", "gpu_metrics": True, "gpu_metrics_hz": 10000, "extra_args": []}
NCU = {"enabled": "ncu" in STAGES, "use_plan": True, "sweep_batch_sizes": [1], "out_tokens": 8,
       "gmu": 0.72, "mns": 64,               # leave ~20 GB free for ncu's kernel replay buffers
       "time_limit_min": 0,                  # >0: the ncu plan skips/stops profiles past this many minutes
       "metrics": ["gpu__time_duration.sum", "dram__bytes_read.sum", "dram__bytes_write.sum",
                   "dram__throughput.avg.pct_of_peak_sustained_elapsed",
                   "sm__throughput.avg.pct_of_peak_sustained_elapsed", "launch__grid_size"]}
DRY_RUN_STAGES = False                    # True: the VM prints every bench/nsys/ncu command without running it

### 1d. Write the run config
Everything above is frozen into `runs/<RUN_ID>/config.json`. The VM scripts read only this file, so the run folder alone tells anyone exactly how the numbers were produced. Re-using a `RUN_ID` keeps earlier results: finished items are skipped.

In [ ]:
RUN_DIR = REPO / "runs" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
cfg_path = RUN_DIR / "config.json"
old = json.loads(cfg_path.read_text()) if cfg_path.exists() else {}
CONFIG = {
    "run_id": RUN_ID,
    "created": old.get("created", datetime.datetime.now().isoformat(timespec="seconds")),
    "gcp": {"project": PROJECT, "zone": ZONES[0], "vm": VM_NAME, "machine_type": MACHINE,
            "provisioning": "SPOT" if SPOT else "STANDARD", "image_family": IMAGE_FAMILY, "boot_disk_gb": BOOT_DISK_GB},
    "vllm": VLLM, "vllm_extra_cli": VLLM_EXTRA_CLI, "env": ENV_VARS,
    "workload": WORKLOAD, "bench": BENCH, "nsys": NSYS, "ncu": NCU,
}
for k in ("bench", "nsys", "ncu"):   # a stage run in an earlier session of this RUN_ID stays marked as run
    CONFIG[k]["enabled"] = CONFIG[k]["enabled"] or old.get(k, {}).get("enabled", False)
if old and old.get("vllm") != VLLM:
    print("WARNING: vLLM args differ from the existing config of this RUN_ID; results would mix settings. "
          "Use a new RUN_ID (or RUN_TAG) for a different configuration.")
cfg_path.write_text(json.dumps(CONFIG, indent=2))
print(f"run folder: {RUN_DIR.relative_to(REPO)}")
print(json.dumps({k: CONFIG[k] for k in ("vllm", "bench", "nsys")}, indent=1))

## 2. Preflight: login, project, quotas
`gcloud` must be logged in and the quotas below must allow 1 H100 and 26 vCPUs. `usage / limit` is shown for each.

In [ ]:
from gpuprof import GCP, GCPError, Preempted

gcp = GCP(PROJECT, ZONES[0], VM_NAME)
print("gcloud account:", gcp.account() or "NOT LOGGED IN -> run `gcloud auth login` in a terminal")
q = gcp.quotas(REGION)
need = {"PREEMPTIBLE_NVIDIA_H100_GPUS" if SPOT else "NVIDIA_H100_GPUS": 1, "GPUS_ALL_REGIONS": 1,
        "CPUS_ALL_REGIONS": 26, "CPUS": 26}
for metric, n in need.items():
    if metric not in q:
        # Newer projects only expose H100 quotas through the Cloud Quotas API, not this legacy listing
        print(f"??   {metric:30s} not reported here -> check Console > IAM & Admin > Quotas (need {n})")
        continue
    usage, limit = q[metric]
    ok = (limit - usage) >= n or (metric == "CPUS" and SPOT and q.get("PREEMPTIBLE_CPUS", (0, 0))[1] >= n)
    print(f"{'OK  ' if ok else 'LOW '} {metric:30s} usage {usage:>5.0f} / limit {limit:>6.0f}   (need {n} free)")
print("VM status:", gcp.status())

## 3. Launch the VM
Creates the VM if it doesn't exist (trying each zone in `ZONES` if one has no Spot capacity), otherwise starts it. Then waits until SSH works and `nvidia-smi` sees the H100 (on first boot the image installs the driver, ~3–5 min), and arms the self-shutdown timer.

In [ ]:
CREATE_SPEC = dict(machine_type=MACHINE, image_family=IMAGE_FAMILY, image_project=IMAGE_PROJECT,
                   disk_gb=BOOT_DISK_GB, zones=ZONES, spot=SPOT)
gcp.ensure_running(CREATE_SPEC)
gcp.set_shutdown_timer(SHUTDOWN_TIMER_MIN)

## 4. Upload code + config, then set up the VM
`upload_code` sends `scripts/`, `requirements-vm.txt` and `runs/<RUN_ID>/config.json` (+ the ncu plan if notebook 02 made one) as one tarball to `~/sillybench` on the VM.

`scripts/vm_setup.sh` is idempotent: it installs `build-essential` (FlashInfer compiles kernels with g++ on first use) and `tmux`, puts `/usr/local/cuda/bin` (nsys, ncu) on PATH, creates `~/venv` with uv, installs the **pinned** `requirements-vm.txt`, and downloads the model weights. On an already-prepared VM it finishes in seconds.

In [ ]:
def upload():
    gcp.upload_code(REPO, RUN_DIR, RUN_ID)

def stage_cmd(stage):
    # Command each stage runs on the VM (inside tmux via scripts/job.sh)
    run = f"runs/{RUN_ID}"
    if stage == "setup":
        return f"bash scripts/vm_setup.sh {VLLM['model']}"
    if stage == "env":
        return f"python scripts/env_capture.py --run-dir {run}"
    return f"python scripts/run_stage.py {stage} --run-dir {run}" + (" --dry-run" if DRY_RUN_STAGES else "")

def run_stage(stage, force=True, cleanup=True):   # force: relaunch even if it finished before (finished items are skipped anyway)
    t0 = time.time()
    try:
        gcp.run_job(RUN_ID, stage, stage_cmd(stage), poll_s=POLL_SECONDS, auto_resume=AUTO_RESUME,
                    on_resume=lambda: (upload(), gcp.set_shutdown_timer(SHUTDOWN_TIMER_MIN)), force=force)
    except Exception:
        # Running cell by cell, a failure stops the notebook before the Shut down cell: don't leave the GPU billing
        if cleanup and ON_FAILURE == "stop":
            print(f"\nstage {stage} FAILED -> pulling what exists (logs) and stopping the VM (ON_FAILURE='stop')")
            try:
                gcp.pull_run(RUN_ID, REPO / "runs")
            except Exception as e:  # noqa: BLE001
                print("pull failed:", e)
            gcp.stop()
        raise
    print(f"stage {stage}: {(time.time()-t0)/60:.1f} min wall")

upload()
if "setup" in STAGES:
    run_stage("setup")      # re-checks everything; seconds when nothing changed

## 5. Environment capture
Records GPU, driver, clocks (`nvidia-smi -q`), CUDA, PyTorch/vLLM/FlashInfer/Triton versions, `pip freeze`, nsys/ncu versions, CPU, machine type and the model's Hugging Face revision into `runs/<RUN_ID>/env/`. This is what makes the results reproducible and comparable across runs.

In [ ]:
if "env" in STAGES:
    run_stage("env")
    print(gcp.ssh(f"cat $HOME/sillybench/runs/{RUN_ID}/env/env.json", echo=False))

## 6. Benchmark sweep (`vllm bench latency`)
For each batch size B and output length O: 3 warm-up + 5 timed runs of `B` prompts × `input_len` tokens. Each item loads the model once (~1 min), so the default sweep of 12 items takes ~20–30 min. Output: `bench/lat_b{B}_in{L}_out{O}.json` plus one log per item in `logs/`.

In [ ]:
if "bench" in STAGES:
    run_stage("bench")

## 7. Nsight Systems captures
For each (batch, mode): `nsys profile --capture-range=cudaProfilerApi --cuda-graph-trace=node --trace=cuda,nvtx,osrt python scripts/prof_run.py ...`. The capture window opens after warm-up, so model loading and graph capture are not recorded. Then `nsys stats` writes the kernel / NVTX summary CSVs and the `.sqlite` that notebook 02 reads.

* `graphs` = production config (CUDA graphs + torch.compile); `eager` = `enforce_eager=True`, used as the A/B baseline.
* The optional `b1_graphs_gpumetrics` capture adds sampled DRAM bandwidth and SM activity rows to the timeline.

In [ ]:
if "nsys" in STAGES:
    run_stage("nsys")

## 8. Nsight Compute
Runs `runs/<RUN_ID>/ncu_plan/ncu_commands.sh` if notebook 02 generated it (re-upload happens automatically below), otherwise a light metric-only sweep. All ncu runs use the production config with `--graph-profiling node`, cap vLLM at `gmu=0.72` so ncu has free memory for its replays, and use `--clock-control none` so kernel times match nsys. Reports are exported to CSV on the VM.

In [ ]:
if "ncu" in STAGES:
    plan = RUN_DIR / "ncu_plan" / "ncu_commands.sh"
    print("ncu plan:", plan.relative_to(REPO) if plan.exists() else "none -> default metric sweep")
    upload()                                  # make sure the latest plan is on the VM
    run_stage("ncu")

## 9. Transfer results to your laptop (verified)
On the VM: write `MANIFEST.sha256` for every file of the run and pack it into one tarball. Copy it, check the tarball's sha256 against the VM's, extract into `runs/<RUN_ID>/`, then re-hash **every file** against the manifest. The VM is only deleted (if `FINAL_ACTION="delete"`) when this verification passes.

In [ ]:
REPORT = gcp.pull_run(RUN_ID, REPO / "runs")
print(json.dumps(REPORT, indent=1))

## 10. Shut down the VM
* `stop` (default, recommended): GPU/CPU billing ends; the boot disk keeps the model, venv, compiled kernels and results, so the next run starts in ~1 min instead of a ~16 min setup. Disk only: ~$0.80/day for 240 GB.
* `delete`: removes the VM and its disk (only allowed after a verified transfer). Next run re-creates it and re-downloads the model. Prefer `stop`; delete only when you're done with the study.

Your H100 quota is a project-level limit and is unaffected by stop or delete. Neither state reserves a GPU: starting a stopped Spot VM needs free capacity again (the notebook retries if there is none).
* `keep`: leaves it running (the auto-shutdown timer still applies).

In [ ]:
def shutdown_vm(report=None):
    action = FINAL_ACTION
    if action == "delete" and not (report and report.get("verified")):
        print("Transfer not verified -> stopping instead of deleting, so no data can be lost.")
        action = "stop"
    if action == "stop":
        gcp.stop()
    elif action == "delete":
        gcp.delete()
    print("VM status now:", gcp.status())
    hrs = gcp.running_hours()
    cost = f" ≈ ${hrs * PRICE_PER_HOUR_USD:.2f}" if PRICE_PER_HOUR_USD else ""
    print(f"VM RUNNING time in this notebook session: {hrs:.2f} h{cost}")

shutdown_vm(REPORT if "REPORT" in globals() else None)

## 11. Analyze this run (notebook 02)
Executes `02_prediction_nsys_analysis.ipynb` on this run folder (no GPU needed) and saves the executed copy with all outputs into `runs/<RUN_ID>/analysis/`. It also writes `runs/<RUN_ID>/ncu_plan/ncu_commands.sh`; run section 1–10 again with the same `RUN_ID` and `STAGES = ["setup", "env", "ncu"]` to execute that plan.

In [ ]:
import os, subprocess
out = RUN_DIR / "analysis" / "02_prediction_nsys_analysis.executed.ipynb"
out.parent.mkdir(exist_ok=True)
env = dict(os.environ, RUN_DIR=str(RUN_DIR))
p = subprocess.run([sys.executable, "-m", "jupyter", "nbconvert", "--to", "notebook", "--execute",
                    str(REPO / "notebooks" / "02_prediction_nsys_analysis.ipynb"), "--output", str(out),
                    "--ExecutePreprocessor.timeout=1200"], env=env, capture_output=True, text=True)
print(p.stderr[-1500:] if p.returncode else f"wrote {out.relative_to(REPO)}")
findings = RUN_DIR / "analysis" / "findings.md"
if findings.exists():
    print(findings.read_text())

## 12. One-click pipeline (alternative to sections 3–10)
Runs every selected stage and **always** pulls whatever finished and shuts the VM down, even if a stage fails or you interrupt the kernel. Use this for unattended runs.

In [ ]:
RUN_PIPELINE = False          # set True and run this cell instead of stepping through 3-10

def run_pipeline():
    report = None
    try:
        gcp.ensure_running(CREATE_SPEC)
        gcp.set_shutdown_timer(SHUTDOWN_TIMER_MIN)
        upload()
        for st in ["setup", "env", "bench", "nsys", "ncu"]:
            if st in STAGES:
                run_stage(st, cleanup=False)          # the finally below pulls + shuts down
    finally:
        try:
            report = gcp.pull_run(RUN_ID, REPO / "runs")
        except Exception as e:  # noqa: BLE001
            print("transfer failed:", e, "-> the VM will be stopped (not deleted); results stay on its disk")
        shutdown_vm(report)
    return report

if RUN_PIPELINE:
    REPORT = run_pipeline()

## Troubleshooting
| Symptom | Fix |
|---|---|
| `ZONE_RESOURCE_POOL_EXHAUSTED` on create/start | No Spot H100 free right now: the notebook tries the other zones / retries; else wait 10–15 min |
| Quota cell shows `LOW` | Request the increase in Console → IAM & Admin → Quotas (see README) |
| A stage fails | The error shows the last 40 log lines; full log in `runs/<RUN_ID>/logs/` on the VM (pull it with section 9) |
| Preempted mid-stage | With `AUTO_RESUME=True` the notebook restarts the VM and relaunches; finished items are skipped |
| `max_num_seqs ... exceeds available Mamba cache blocks` | Lower `max_num_seqs` in 1b (≤ 358 at `gpu_memory_utilization=0.92`) |
| `cannot execute 'cc1plus'` | `build-essential` missing: rerun the setup cell |
| ncu is very slow | Lower `gmu`, fewer targets; check progress with `tmux attach -t ncu` on the VM |
| Watch a job live | `gcloud compute ssh h100-prof --zone us-central1-a -- tmux attach -t <stage>` (detach: Ctrl+B, D) |